# Notebook 02 — HF_v3 Global Validation

Reproduces the global Harmony Field correlation analysis.

**Formula:**
```
HF_weighted = w_h * hf_harmony + w_t * hf_tension + w_c * hf_conjunction
weights: w_h=-1.0, w_t=-1.0, w_c=+2.5
```

**Inputs:** `data/corpus/events_public.csv`  
**Outputs:** `figures/hf_valence_distribution.png`, `data/results/correlation_v3_global.json`

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from pathlib import Path

REPO = Path('.')

events = pd.read_csv(REPO / 'data/corpus/events_public.csv')

# Valence numeric: positive=1, negative=-1, neutral=0
valence_map = {'positive': 1, 'negative': -1, 'neutral': 0}
events['valence_num'] = events['valence'].map(valence_map)

print(f'Total events: {len(events)}')
print(f'Valence counts: {events["valence"].value_counts().to_dict()}')

In [ ]:
# Pearson r — all events
r_all, p_all = stats.pearsonr(events['hf_weighted'], events['valence_num'])
print(f'Pearson r (all): {r_all:.4f}, p={p_all:.4f}')

# Non-neutral only
non_neu = events[events['valence'] != 'neutral']
r_nn, p_nn = stats.pearsonr(non_neu['hf_weighted'], non_neu['valence_num'])
print(f'Pearson r (non-neutral): {r_nn:.4f}, p={p_nn:.4f}')

# Cohen's d
pos = events[events['valence']=='positive']['hf_weighted'].values
neg = events[events['valence']=='negative']['hf_weighted'].values

n1, n2 = len(pos), len(neg)
s_pooled = np.sqrt(((n1-1)*pos.std(ddof=1)**2 + (n2-1)*neg.std(ddof=1)**2) / (n1+n2-2))
d = (pos.mean() - neg.mean()) / s_pooled

print(f"Cohen's d: {d:.3f}")
print(f'Mean HF+ = {pos.mean():.3f}')
print(f'Mean HF- = {neg.mean():.3f}')

In [ ]:
# Violin plot
fig, ax = plt.subplots(figsize=(8, 4))

data_groups = [
    events[events['valence']=='positive']['hf_weighted'].values,
    events[events['valence']=='negative']['hf_weighted'].values,
    events[events['valence']=='neutral']['hf_weighted'].values,
]

parts = ax.violinplot(data_groups, positions=[1,2,3], showmeans=True)
for pc, col in zip(parts['bodies'], ['#C9A84C','#8B2020','#555555']):
    pc.set_facecolor(col)
    pc.set_alpha(0.6)

ax.set_xticks([1,2,3])
ax.set_xticklabels(['Positive', 'Negative', 'Neutral'])
ax.set_ylabel('HF Weighted')
ax.set_title(f"HF_v3 by Valence — r={r_all:.3f}, Cohen's d={d:.3f}")
plt.tight_layout()
plt.show()

In [ ]:
# Load and display stored results
with open(REPO / 'data/results/correlation_v3_global.json') as f:
    stored = json.load(f)

print('Stored results (from build script):')
for k, v in stored.items():
    print(f'  {k}: {v}')